In [367]:
# !pip install openpyxl

In [368]:
import random
import string
from datetime import datetime, timedelta
import pandas as pd
import re
import numpy as np


In [369]:
def gerar_cpf():
    """
    Gera CPF formatado.
    Exemplo: 123.456.789-01
    """
    nums = [random.randint(0, 9) for _ in range(11)]

    return (
        f"{nums[0]}{nums[1]}{nums[2]}."
        f"{nums[3]}{nums[4]}{nums[5]}."
        f"{nums[6]}{nums[7]}{nums[8]}-"
        f"{nums[9]}{nums[10]}"
    )

In [370]:
def gerar_cnpj():
    """
    Gera CNPJ formatado.
    Exemplo: 12.345.678/0001-99
    """
    nums = [random.randint(0, 9) for _ in range(14)]

    return (
        f"{nums[0]}{nums[1]}."
        f"{nums[2]}{nums[3]}{nums[4]}."
        f"{nums[5]}{nums[6]}{nums[7]}/"
        f"{nums[8]}{nums[9]}{nums[10]}{nums[11]}-"
        f"{nums[12]}{nums[13]}"
    )

In [371]:
def gerar_dados_sinteticos(qtd_registros=100):
    """
    Gera dados sintéticos de chamados.

    Campos:
    - id_task
    - cd_pin
    - cd_modelo
    - nr_contrato
    - cd_unico_cliente
    - nr_cpf_cnpj
    - nr_ano_mes
    """

    modelos = [
        "9492", "90", "511",
        "201", "946", "251"
    ]

    pins = [
          "2050", "2051", "3000", 
          "3050", "5032", "3054"]

    seguimentos = [
          "3", "9", "256", "11"
    ]


    dados = []

    for i in range(qtd_registros):

        # Decide aleatoriamente se será CPF ou CNPJ
        if random.choice([True, False]):
                nr_cpf_cnpj = gerar_cpf()
        else:
            nr_cpf_cnpj = gerar_cnpj()

        # Gera ano/mês aleatório entre 2020 e hoje
        data_inicio = datetime(2026, 2, 1)
        data_fim = datetime.now()

        dias_range = (data_fim - data_inicio).days
        data_random = data_inicio + timedelta(days=random.randint(0, dias_range))

        nr_ano_mes = data_random.strftime("%Y%m")

        registro = {
            "ID_TASK": i + 9435,
            "CD_UNIDADE": random.randint(111111, 999999),
            "CD_PIN": random.choice(pins),
            "CD_MODELO": random.choice(modelos),
            "NR_CONTRATO": f"66000{random.randint(1000000, 9999999)}",
            "CD_CLI": random.randint(10000000, 99999999),
            "CD_SEGMENTO": random.choice(seguimentos),
            "CPF_CNPJ": nr_cpf_cnpj,
            "ANOMES": "202604"
        }

        dados.append(registro)

    return dados

In [372]:
def aplicar_ruido_nos_dados(lista_dados, percentual_ruido=0.20):
    """Consome a lista de dados e aplica ruídos/inversões de forma INDEPENDENTE

    nos registros, baseando-se no percentual escolhido.
    """
    bancos = ["033"]

    for registro in lista_dados:

        # --- ERRO 1: CONTRATO RECEBE BANCO + AGÊNCIA ---
        # Acontece de forma independente para a % escolhida
        if random.random() < percentual_ruido:
            cod_banco = random.choice(bancos)
            cod_agencia = f"{random.randint(1, 9999):04d}"
            contrato_original = registro["NR_CONTRATO"]
            registro["NR_CONTRATO"] = (
                f"{cod_banco}{cod_agencia}{contrato_original}"
            )

        # --- ERRO 2: UNIDADE (PERNUMPER) FICA SUJA ---
        # Também independente. Pode acontecer junto com o erro acima, ou sozinho
        if random.random() < percentual_ruido:
            chance_sujeira = random.random()
            if chance_sujeira < 0.50:  # 50% de chance de virar 0, 1, 0000
                registro["CD_CLI"] = random.choice(
                    ["0", "1", "0000", "00000000"]
                )
            else:  # 50% de chance de virar texto alfanumérico misto
                tamanho = random.randint(4, 10)
                registro["CD_CLI"] = "".join(
                    random.choice(string.ascii_uppercase + string.digits)
                    for _ in range(tamanho)
                )

        # --- ERRO 3: INVERSÃO DE COLUNAS ---
        # Troca os valores de lugar de forma independente
        if random.random() < percentual_ruido:
            registro["NR_CONTRATO"], registro["CD_CLI"] = (
                registro["CD_CLI"],
                registro["NR_CONTRATO"],
            )

    return lista_dados

In [373]:
def capturarContrato(x):
    contrato = re.findall(r'66\d+', x)
    return contrato[0] if contrato else None

In [374]:
def validar_CLI(x):
    return bool(re.fullmatch(r'[A-Za-z0-9]{8}', str(x["CD_CLI"])))

In [375]:
def verificarGradePesos(x):
    dfGrade = pd.read_excel("./database/gold/gradePesos.xlsx")
    
    # Adicione estas duas linhas para forçar a tipagem para string
    dfGrade["modelo"] = dfGrade["modelo"].astype(str)
    dfGrade["pin"] = dfGrade["pin"].astype(str)
    
    mask = (
        (dfGrade["modelo"] == str(x["CD_MODELO"])) &
        (dfGrade["pin"] == str(x["CD_PIN"]))
    )
    return not dfGrade.loc[mask].empty

In [376]:
def verificarModelo(x):
    dfModelo = pd.read_excel("./database/gold/modeloAngariacao.xlsx")
    
    # Forçando tipagem para string
    dfModelo["modelo"] = dfModelo["modelo"].astype(str)
    dfModelo["segmento"] = dfModelo["segmento"].astype(str)
    
    mask = (
        (dfModelo["modelo"] == str(x["CD_MODELO"])) &
        (dfModelo["segmento"] == str(x["CD_SEGMENTO"]))
    )
    return not dfModelo.loc[mask].empty

In [377]:
def motorCorpao(dfChamados):
    dfChamados["regexContrato"] = dfChamados["NR_CONTRATO"].apply(lambda x: capturarContrato(str(x)))
    dfCorpao = dfChamados.loc[~dfChamados["regexContrato"].isna(), ["CD_UNIDADE", "CD_PIN", "CD_MODELO", "CD_SEGMENTO", "regexContrato", "CPF_CNPJ", "CD_CLI", "ANOMES"]]

    dfCorpao["CD_ANG"] = dfCorpao.apply(
        lambda x: x["CD_UNIDADE"] if random.random() < 0.8 else random.randint(100000, 999999),
        axis=1
    )

    dfCorpao["pinModelo"] = dfCorpao.apply(lambda x: verificarGradePesos(x),axis=1)
    dfCorpao = dfCorpao[dfCorpao["pinModelo"] == True]

    dfCorpao["pinSegmento"] = dfCorpao.apply(lambda x: verificarModelo(x),axis=1)
    dfCorpao = dfCorpao[dfCorpao["pinSegmento"] == True]

    dfCorpao["CD_CLI_VALIDO"] = dfCorpao.apply(lambda x: validar_CLI(x),axis=1)
    dfCorpao = dfCorpao[dfCorpao["CD_CLI_VALIDO"] == True]

    dfCorpao["ANOMES"] = dfCorpao["ANOMES"].apply(lambda x: random.choice(["202603","202604","202605"]) )

    dfCorpao = dfCorpao.drop(columns=['pinModelo', 'pinSegmento', 'CD_CLI_VALIDO'])
   
    return dfCorpao

In [381]:
# 1. Gera dados limpos
dados_originais = gerar_dados_sinteticos(500)

# 2. Aplica ruído
dados_com_ruido = aplicar_ruido_nos_dados(dados_originais, percentual_ruido=0.1)
df = pd.DataFrame(dados_com_ruido)
df.to_excel("./database/Bronze/chamados.xlsx")
motorCorpao(df).to_excel("./database/Bronze/corpao.xlsx")


dadosTreinamento = aplicar_ruido_nos_dados(gerar_dados_sinteticos(5000), percentual_ruido=0.1)
dadosTreinamento = pd.DataFrame(dadosTreinamento)
dadosTreinamento.to_excel("./database/Bronze/chamadosTreinamento.xlsx")
motorCorpao(dadosTreinamento).to_excel("./database/Bronze/corpaoTreinamento.xlsx")
